# KAL + GMS — end-to-end demo

Ask a question in **English** → **KAL** aggregates heterogeneous sources into
triples → **GMS** trains a geometric knowledge graph → a **grounded answer /
verdict / 3D route**.

> Run cells top-to-bottom. Sections 1–5 need only a KnowlytiX license.
> Sections 6–7 also need your own Anthropic key. Section 8 optionally needs
> your own hosted Neo4j database. See the repository `README.md` for setup.


In [ ]:
# Load secrets from Colab Secrets or environment variables (for local/CI runs).
from pathlib import Path
import os

try:
    from google.colab import userdata
except ImportError:
    userdata = None

def get_secret(name: str) -> str | None:
    value = os.environ.get(name)
    if value:
        return value.strip()
    if userdata is not None:
        try:
            value = userdata.get(name)
        except Exception:
            value = None
    return str(value).strip() if value else None

license_key = get_secret("KNOWLYTIX_LICENSE_KEY")
if not license_key or any(character.isspace() for character in license_key):
    raise RuntimeError(
        "Provide a valid KNOWLYTIX_LICENSE_KEY in Colab Secrets or the environment."
    )

os.environ["KNOWLYTIX_LICENSE_KEY"] = license_key
runtime_license_file = Path.home() / ".knowlytix" / "license.key"
runtime_license_file.parent.mkdir(parents=True, exist_ok=True)
runtime_license_file.write_text(license_key + "\n", encoding="utf-8")

print("KnowlytiX license loaded without displaying it.")
print("Runtime license file:", runtime_license_file)


In [ ]:
# Install into Colab's fast local disk; persist only pip downloads in Google Drive.
import os
import platform
import subprocess
import sys
from pathlib import Path

if sys.version_info[:2] != (3, 12):
    raise RuntimeError(f"This demo requires Python 3.12; found {platform.python_version()}.")
if platform.system() != "Linux" or platform.machine().lower() not in {"x86_64", "amd64"}:
    raise RuntimeError(
        f"The bundled GMS extensions require Linux x86-64; found "
        f"{platform.system()} {platform.machine()}."
    )

PACKAGE_SPEC = (
    "knowlytix-colab-demo[kal-gms] @ "
    "git+https://github.com/knowlytix/colab-demo.git"
)
INSTALL_REVISION = "1.0.1"  # Bump when bundled native artifacts change.

# The cache path is relative to My Drive and independent of the notebook location.
DRIVE_FOLDER = Path("KnowlytiX") / "colab-demo"
USE_DRIVE_PIP_CACHE = True
REFRESH_PACKAGE = False
runtime_marker = Path("/content/.knowlytix-kal-gms-installed")

if userdata is not None:  # Google Colab
    if USE_DRIVE_PIP_CACHE:
        from google.colab import drive

        drive.mount("/content/drive", force_remount=False)
        if DRIVE_FOLDER.is_absolute() or ".." in DRIVE_FOLDER.parts:
            raise ValueError("DRIVE_FOLDER must be a safe path relative to My Drive.")
        cache_dir = Path("/content/drive/MyDrive") / DRIVE_FOLDER / "pip-cache"
        cache_dir.mkdir(parents=True, exist_ok=True)
        os.environ["PIP_CACHE_DIR"] = str(cache_dir)
        print("Persistent pip cache:", cache_dir)

    marker_value = f"{PACKAGE_SPEC}\n{INSTALL_REVISION}\n"
    marker_matches = (
        runtime_marker.is_file()
        and runtime_marker.read_text(encoding="utf-8") == marker_value
    )
    if REFRESH_PACKAGE or not marker_matches:
        command = [
            sys.executable, "-m", "pip", "install",
            "--prefer-binary", "--upgrade-strategy", "only-if-needed",
        ]
        if REFRESH_PACKAGE:
            command.append("--upgrade")
        subprocess.check_call([*command, PACKAGE_SPEC])
        runtime_marker.write_text(marker_value, encoding="utf-8")
        print("Installed KAL + GMS dependencies on Colab's local runtime disk.")
    else:
        print("KAL + GMS dependencies are already installed in this runtime.")
else:  # Local Python or GitHub Actions
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "--prefer-binary",
        "--upgrade-strategy",
        "only-if-needed",
        PACKAGE_SPEC,
    ])
    print("Installed KAL + GMS dependencies in the active Python environment.")


## 0. Preflight — license, runtime, packages, and fixtures

The first two cells load the license from Colab Secrets and install this
repository from GitHub. This check confirms that the packaged demo fixtures
are available before training any stores.


In [ ]:
from knowlytix_demo.preflight import preflight
from knowlytix_demo.resources import available_fixtures

preflight(require_llm=False)
print("Bundled fixtures:", ", ".join(available_fixtures()))


## 1. The spine — KAL triples → GMS verdict

Load triple data via KAL, train a geometric store, and score a claim. Lower
score = more plausible; the geometry separates a true claim from a false one.

In [ ]:
from knowlytix_demo.build_store_from_triples import load_or_build_store_from_triples
from knowlytix_demo.kal_loader import load_triples  # async; Colab supports top-level await

triples = await load_triples("finance_seed", "fixtures/finance_seed.jsonl")
fin_store = load_or_build_store_from_triples(triples, store_path="_store_nb_fin")
print("true ", fin_store.score_triple("Foxglove BioSciences, Inc.", "reports_revenue", "$96.8B FY2024"))
print("false", fin_store.score_triple("Foxglove BioSciences, Inc.", "reports_revenue", "Ana Patel"))


## 2. Lane 1 — extract triples from an unstructured document

KAL covers structured triples; for prose/tables, the parser extracts triples
that merge into the same graph. (General but lossy — see gap **G3**: dumping
lossy triples into a curated graph blurs verdicts.)

In [ ]:
from knowlytix_demo.extractor import extract_triples_from_file

extracted, stats = extract_triples_from_file("fixtures/finance_excerpt.md", mode="regex")
print(f"{stats.get('triples')} triples extracted; {len(extracted)} after dropping provenance")
extracted[:5]


## 3. Cross-source multi-hop — an answer no single source can give

Two *partial* sources share a hand-aligned join entity. Neither alone answers
"what risk tier applies to the tool a data engineer uses?"; the KAL-**unified**
graph does. (Pure geometry — chained `link_predict`, no LLM.)

In [ ]:
from knowlytix_demo.gms_query import multi_hop

roles = await load_triples("roles", "fixtures/gov_roles.jsonl")
risk = await load_triples("risk", "fixtures/gov_risk.jsonl")
chain = ["uses_tool", "has_risk_tier"]

xsrc_store = load_or_build_store_from_triples(roles + risk, store_path="_store_nb_xsrc")
for label, triples_for_source in [("roles only", roles), ("risk only", risk)]:
    store = load_or_build_store_from_triples(
        triples_for_source, store_path="_store_nb_" + label.split()[0]
    )
    answer, _ = multi_hop(store, "data_engineer", chain)
    print(f"{label:10}: " + ("cannot answer" if answer is None else answer))
answer, _ = multi_hop(xsrc_store, "data_engineer", chain)
print(f"unified   : {answer}")


## 4. Governance lane — multi-hop + a policy verdict

A curated governance graph (the upstream model-risk vocab). We build it **once**
and reuse it for the multi-hop answer, the 3D route, and the NL Q&A below.

In [ ]:
gov = await load_triples("governance_seed", "fixtures/governance_seed.jsonl")
gov_store = load_or_build_store_from_triples(gov, store_path="_store_nb_gov")

ans, hops = multi_hop(gov_store, "data_engineer", ["uses_tool", "has_risk_tier"])
for h in hops:
    print(f"  {h.head} --{h.relation}--> {h.tail}")
print("answer:", ans)

print("\nverdict (lower = more plausible):")
print("  true ", gov_store.score_triple(
    "regulated_lending_model", "requires_check", "fair_lending_review"))
print("  false", gov_store.score_triple(
    "regulated_lending_model", "requires_check", "model_validation"))

## 5. Two roles, one chain — GMS makes the gap visible

We run the **same** multi-hop question (`uses_tool → has_risk_tier`) for two
roles against the cross-source `xsrc_store`:

* `data_engineer` — has a real `uses_tool → jupyter_notebook → has_risk_tier
  → tier_2` path. Every retrieved hop is a triple in the training set, so
  `trace.path_valid` is **True**: this is a *grounded* answer.
* `model_risk_officer` — has **no** `uses_tool` edge at all. The geometry
  still returns a nearest tail under the relation cap (that's what
  `link_predict` does), but `trace.path_valid` is **False**: GMS knows the
  geometric tail is *phantom* — there's no source triple to back it up.

That contrast is the headline: **GMS exposes "I don't actually know" instead
of presenting a confident geometric guess as if it were ground truth.**


In [ ]:
from IPython.display import display
from knowlytix_demo.viz_sphere import build_figure, load_store, run_query

loaded = load_store("_store_nb_xsrc")
chain = ["uses_tool", "has_risk_tier"]

def _summarise(label: str, trace) -> None:
    verdict = "YES — every hop is a real triple" if trace.path_valid \
              else "NO  — geometry guessed a phantom"
    print(f"[{label}]")
    print(f"  retrieved (geometric): {trace.retrieved}  ({trace.retrieved_verdict})")
    print(f"  path_valid (graph)   : {verdict}")
    for hop, ok in zip(trace.nearest_per_hop, trace.path_valid_per_hop, strict=True):
        marker = "OK  " if ok else "MISS"
        print(f"    {marker} {hop.rotor:18s} -> {hop.name}")

for role in ("data_engineer", "model_risk_officer"):
    trace = run_query(loaded, role, chain)
    _summarise(role, trace)
    display(build_figure(loaded, trace))


## 6. Grounded vs inferred vs LLM-only — same question, three answers  *(needs a provider key)*

For each role we ask the **same** English question and produce three answers:

| Mode | What it does | When it speaks |
|---|---|---|
| **GMS grounded** | parse → multi-hop → `compose_answer` (gated on `path_valid`) | Only when every retrieved hop is a real edge |
| **GMS inferred** | walk the source's *actual* outgoing edges → hedged LLM answer | When the asked chain has no support; output is prefixed `INFERENCE — ` |
| **LLM-only** | `answer_without_graph` — no graph, no vocab, no facts | Always (the hallucination control) |

For `data_engineer` the grounded mode wins. For `model_risk_officer` the
graph refuses (no `uses_tool` edge), the inferred mode hedges over
`can_approve`, and the LLM-only mode will confidently invent a tier — exactly
the hallucination GMS is designed to catch.


In [ ]:
# Sections 6–7 make external Anthropic API calls.
anthropic_key = get_secret("ANTHROPIC_API_KEY")
if not anthropic_key:
    raise RuntimeError("Provide ANTHROPIC_API_KEY in Colab Secrets or the environment.")
os.environ["ANTHROPIC_API_KEY"] = anthropic_key
llm_model = get_secret("GMS_LLM_MODEL")
if llm_model:
    os.environ["GMS_LLM_MODEL"] = llm_model
else:
    os.environ["GMS_LLM_MODEL"] = "anthropic/claude-sonnet-4-6"

from knowlytix_demo.gms_query import is_path_valid
from knowlytix_demo.nl_query import (
    answer_without_graph,
    available_edges,
    compose_answer,
    compose_inference,
    parse_question,
)

preflight(require_llm=True)
vocab_e = list(gov_store.adapter.entity_to_idx)
vocab_r = list(gov_store.adapter.relation_to_idx)

QUESTIONS = {
    "data_engineer": "What risk tier applies to the tool a data engineer uses?",
    "model_risk_officer": "What risk tier applies to the tool a model risk officer uses?",
}

# Keep successful paid responses in memory so rerunning sections 6–7 does not
# repeat identical API calls. A runtime restart intentionally clears the cache.
_LLM_RESULTS = globals().get("_LLM_RESULTS", {})
def _llm_once(operation, inputs, call):
    key = (os.environ["GMS_LLM_MODEL"], operation, inputs)
    if key not in _LLM_RESULTS:
        _LLM_RESULTS[key] = call()
    return _LLM_RESULTS[key]

for role, question in QUESTIONS.items():
    print(f"\n=== {role} ===")
    print(f"Q: {question}")
    source, relations = _llm_once(
        "parse", question, lambda: parse_question(question, vocab_e, vocab_r)
    )
    print(f"  parsed -> ({source}, {relations})")
    answer, hops = multi_hop(gov_store, source, relations)
    valid = is_path_valid(hops, gov)
    print(f"  path_valid    : {valid}")
    grounded = _llm_once(
        "grounded", (question, answer, valid),
        lambda: compose_answer(question, answer, hops, path_valid=valid),
    )
    print(f"  GMS grounded  : {grounded}")
    if not valid:
        inferred = _llm_once(
            "inferred", (question, source),
            lambda: compose_inference(question, source, available_edges(gov, source)),
        )
        print(
            "  GMS inferred  : "
            f"{inferred}"
        )
    llm_only = _llm_once("llm-only", question, lambda: answer_without_graph(question))
    print(f"  LLM-only      : {llm_only}")


## 7. Phrasing-invariance vs LLM drift  *(needs an Anthropic key)*

Fan each question from §6 into a small deterministic factor matrix of persona,
style, and length variants. For every phrasing we compare the grounded GMS
verdict with an LLM-only response. This keeps the experiment self-contained
without requiring the optional harness package.


In [ ]:
from knowlytix_demo.nl_query import rephrase_question

design = [
    {"role": "auditor", "style": "direct", "length": "short"},
    {"role": "engineer", "style": "conversational", "length": "short"},
    {"role": "executive", "style": "formal", "length": "long"},
    {"role": "analyst", "style": "technical", "length": "long"},
]

def _norm(text: str) -> str:
    return " ".join(text.lower().split())

for role, base_q in QUESTIONS.items():
    print(f"\n=== {role} ===")
    parses: set[tuple[str, tuple[str, ...]]] = set()
    gms_verdicts: set[str] = set()
    llm_responses: set[str] = set()
    for factors in design:
        factor_key = tuple(sorted(factors.items()))
        variant = _llm_once(
            "rephrase", (base_q, factor_key),
            lambda: rephrase_question(base_q, factors),
        )
        src, rels = _llm_once(
            "parse", variant, lambda: parse_question(variant, vocab_e, vocab_r)
        )
        answer, hops = multi_hop(gov_store, src, rels)
        valid = is_path_valid(hops, gov)
        verdict = answer if valid else "REFUSED"
        llm_text = _llm_once(
            "llm-only", variant, lambda: answer_without_graph(variant)
        )
        parses.add((src, tuple(rels)))
        gms_verdicts.add(verdict)
        llm_responses.add(_norm(llm_text))
        print(f"  {variant!r}")
        print(f"    GMS verdict : {verdict}")
        print(f"    LLM-only    : {llm_text[:90]}")
    print(
        f"\n  {len(design)} phrasings -> {len(parses)} parse(s), "
        f"{len(gms_verdicts)} GMS verdict(s), "
        f"{len(llm_responses)} distinct LLM-only response(s)"
    )


## 8. Progressive enrichment — federation extends grounding, doesn't fabricate it  *(needs Neo4j)*

This optional section uses a Neo4j database reachable from Colab. Add
`NEO4J_URI`, `NEO4J_USERNAME`, and `NEO4J_PASSWORD` to Colab Secrets before
running it. Use a disposable database because the demo adds nodes and
relationships with idempotent `MERGE` statements.


In [ ]:
from dataclasses import replace

import numpy as np
import torch
from IPython.display import display
from knowlytix_demo.kal_loader import load_triples_neo4j
from neo4j import GraphDatabase

from knowlytix.knowledge.config import DocGMSConfig

NEO4J = (
    get_secret("NEO4J_URI"),
    get_secret("NEO4J_USERNAME"),
    get_secret("NEO4J_PASSWORD"),
)
if not all(NEO4J):
    raise RuntimeError(
        "Provide NEO4J_URI, NEO4J_USERNAME, and NEO4J_PASSWORD in Colab Secrets."
    )

def merge_triples(driver, triples):
    """Batch idempotent writes into one query per safe relationship type."""
    by_relation = {}
    for head, relation, tail in triples:
        if not relation.isidentifier():
            raise ValueError(f"Unsafe Neo4j relationship type: {relation!r}")
        by_relation.setdefault(relation, []).append({"h": head, "t": tail})
    with driver.session() as session:
        for relation, rows in by_relation.items():
            session.run(
                f"UNWIND $rows AS row MERGE (a {{name:row.h}}) "
                f"MERGE (b {{name:row.t}}) MERGE (a)-[:`{relation}`]->(b)",
                rows=rows,
            ).consume()

# Seed the RISK half into Neo4j (roles + compliance stay JSONL); MERGE is idempotent.
_risk_seed = await load_triples("risk", "fixtures/gov_risk.jsonl")
_drv = GraphDatabase.driver(NEO4J[0], auth=(NEO4J[1], NEO4J[2]))
_drv.verify_connectivity()
merge_triples(_drv, _risk_seed)
_drv.close()

# Pull each source through its own KAL adapter (JSONL + Neo4j).
roles  = await load_triples("roles-jsonl", "fixtures/gov_roles.jsonl")
risk   = await load_triples_neo4j(uri=NEO4J[0], user=NEO4J[1], password=NEO4J[2])
checks = await load_triples("compliance-jsonl", "fixtures/gov_extras.jsonl")

# Per-stage retrieval: train a small GMS store from the federated triples and
# probe `source -> chain`. Each stage prints path_valid + per-hop OK/MISS so
# the audience sees the same calibrated gate from §5. We use 600 epochs (vs
# the 200 default) because the per-stage graph is small — the cap-aware
# multi-hop is reliable at 600 across all five stages, but a few of them
# (notably +compliance and +mro_ops) under-train at 200 and would surface as
# false-negative refusals. Training is still <10s per stage on CPU.
_TRAIN = DocGMSConfig()
_TRAIN = replace(_TRAIN, train=replace(_TRAIN.train, epochs=600))

def enrich(label, triples, chain, source="data_engineer"):
    torch.manual_seed(2)        # the small-graph viz transport is seed-sensitive;
    np.random.seed(2)           # a fixed seed keeps the staged retrievals deterministic
    load_or_build_store_from_triples(
        triples, config=_TRAIN, store_path="_store_nb_" + label
    )
    loaded = load_store("_store_nb_" + label)
    trace = run_query(loaded, source, chain)
    verdict = "GROUNDED" if trace.path_valid else "REFUSED (phantom path)"
    print(f"[{label:14}] {len(triples):2} triples | {source:18s} -> "
          f"{trace.retrieved:20s} | path_valid: {verdict}")
    for hop, ok in zip(trace.nearest_per_hop, trace.path_valid_per_hop, strict=True):
        marker = "OK  " if ok else "MISS"
        print(f"    {marker} {hop.rotor:18s} -> {hop.name}")
    return build_figure(loaded, trace)


In [ ]:
# Stage 1-3: same DE question; the graph grows one KAL adapter at a time.
display(enrich("roles",         roles,                  ["uses_tool"]))                       # 1-hop: tool, no tier yet
display(enrich("+neo4j_risk",   roles + risk,           ["uses_tool", "has_risk_tier"]))      # 2-hop completes to tier_2
display(enrich("+compliance",   roles + risk + checks,  ["uses_tool", "has_risk_tier"]))      # same answer, richer context

# Stage 4: same federation, model_risk_officer. No source has a uses_tool edge
# from MRO — federation expands what GMS can ground, it doesn't fabricate.
display(enrich("+all (MRO)",    roles + risk + checks,  ["uses_tool", "has_risk_tier"],
               source="model_risk_officer"))


### 8b. Live update — an MRO ops source comes online

Stage 4 refused because no source contained a `uses_tool` edge from
`model_risk_officer`. Now an **MRO operations team** brings its tooling
registry online: a new KAL source (`gov_mro_ops.jsonl`) declaring the
validation workbench MRO uses and its tier. We seed it into the **same**
Neo4j (treating the federation update as a live ingest) and re-pull. The
chain that was refused at stage 4 grounds immediately.


In [ ]:
# A new KAL source comes online: the MRO ops team's tooling registry.
mro_ops = await load_triples("mro_ops-jsonl", "fixtures/gov_mro_ops.jsonl")
print(f"mro_ops source: {len(mro_ops)} triple(s)")
for h, r, t in mro_ops:
    print(f"  {h} --{r}--> {t}")

# Federate it into the same Neo4j as a live ingest (MERGE is idempotent, so
# re-runs are safe).
_drv = GraphDatabase.driver(NEO4J[0], auth=(NEO4J[1], NEO4J[2]))
merge_triples(_drv, mro_ops)
_drv.close()

# Re-pull the Neo4j risk source — it now contains the MRO ops edges too.
risk_after = await load_triples_neo4j(uri=NEO4J[0], user=NEO4J[1], password=NEO4J[2])
print(f"\nNeo4j now contains {len(risk_after)} edge(s) (was {len(risk)} before).")

# Stage 5: same question, same chain, same MRO source — the federation just
# learned the missing edge.
display(enrich("+mro_ops",     roles + risk_after + checks,
               ["uses_tool", "has_risk_tier"], source="model_risk_officer"))


---
**Caveats (honest):** the geometric core is licensed (open-core); cross-source
alignment is hand-aligned here (**G1**); "live" updates retrain at small scale
(**G2**); bulk lossy extraction blurs verdicts (**G3**). The calibrated
PASS/REJECT verdict (**G4**) shows up twice in this notebook: as `path_valid`
in §5–§8 (per-hop edge membership against the source triples) and as the
trained tension channel in the GMS core (the threshold is calibrated from
the graph). See the repository `README.md` for installation, credential,
and execution-order guidance.
